In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
confound_strength = 0.15
seed = 83
lookback = 2
hidden_dims = {'C'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed, confound_strength=confound_strength)

# for eval: corrupted W, O hidden
eval_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=False, seed=seed, confound_strength=confound_strength)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = HumanoidMazePCH(num_steps=small_steps, seed=seed, confound_strength=confound_strength)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

{'A0',
 'A1',
 'E0',
 'E1',
 'H0',
 'H1',
 'J0',
 'J1',
 'P0',
 'P1',
 'V0',
 'V1',
 'W0',
 'W1',
 'X0'}

## Expert Trajectories

In [7]:
# for eval: corrupted W, O shown
expert_traj_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, confound_strength=confound_strength)
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/humanoidmaze_medium_expert_finetuned.pt'
expert_ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

expert_action_bounds = (expert_ckpt['action_bounds_low'], expert_ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=expert_ckpt['input_dim'],
    action_dim=expert_ckpt['num_actions'],
    hidden_dim=256,
    num_blocks=expert_ckpt['num_blocks'],
    dropout=expert_ckpt['dropout'],
    layernorm=expert_ckpt['layernorm'],
    final_tanh=expert_ckpt['final_tanh'],
    action_bounds=expert_action_bounds,
).to(device)

expert_model.load_state_dict(expert_ckpt['state_dict'])
expert_model.eval()

expert_slots = expert_ckpt['slots']
expert_Z_trim = expert_ckpt['Z_trim']
expert_dims = expert_ckpt['dims']
expert_lookback = expert_ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, expert_slots, expert_Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)
expert_num_eval_eps = 500

records = collect_imitator_trajectories(
    env=expert_traj_env,
    policies=expert_policies,
    num_episodes=expert_num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(records)

Starting episode 1/500...


  Episode 1 ended at step 1290 (terminated: True, truncated: False).
Starting episode 2/500...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/500...


  Episode 3 ended at step 535 (terminated: True, truncated: False).
Starting episode 4/500...


  Episode 4 ended at step 1783 (terminated: True, truncated: False).
Starting episode 5/500...


...

  Episode 497 ended at step 2000 (terminated: False, truncated: True).
Starting episode 498/500...


  Episode 498 ended at step 991 (terminated: True, truncated: False).
Starting episode 499/500...


  Episode 499 ended at step 2000 (terminated: False, truncated: True).
Starting episode 500/500...


  Episode 500 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


830459

In [8]:
dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    'V': 3,
    # 'C': 3,
    'J': 27,
    'W': 2,
    'X': 21
}

## Training

In [9]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 30
num_blocks = 4
epochs = 100
dropout = 0.0

In [10]:
nbc_model, nbc_slots, nbc_Z_trim = train_single_policy_long_horizon(
    records,
    naive_Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions=train_env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(train_env.action_space.low, train_env.action_space.high)
)

nbc_policy = shared_policy_fn_long_horizon(nbc_model, nbc_slots, nbc_Z_trim, continuous=True, device=device)
nbc_policies = make_shared_policy_dict(nbc_policy)

[LongHorizon] Epoch 1: train loss = 0.051679, val loss = 0.028467.


[LongHorizon] Epoch 2: train loss = 0.024285, val loss = 0.021585.


[LongHorizon] Epoch 3: train loss = 0.020170, val loss = 0.019223.


[LongHorizon] Epoch 4: train loss = 0.018339, val loss = 0.017874.


[LongHorizon] Epoch 5: train loss = 0.017220, val loss = 0.016961.


...

[LongHorizon] Epoch 96: train loss = 0.007715, val loss = 0.009003.


[LongHorizon] Epoch 97: train loss = 0.007692, val loss = 0.008948.


[LongHorizon] Epoch 98: train loss = 0.007681, val loss = 0.008940.


[LongHorizon] Epoch 99: train loss = 0.007652, val loss = 0.008951.


[LongHorizon] Epoch 100: train loss = 0.007623, val loss = 0.009095.


## Save Model

In [11]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'nbc_hummed_c15_s83.pt')

checkpoint = {
    "state_dict": nbc_model.state_dict(),
    "slots": nbc_slots,
    "Z_trim": nbc_Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": train_env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": dropout,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": eval_env.action_space.low,
    "action_bounds_high": eval_env.action_space.high,
    "input_dim": int(nbc_model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/nbc_hummed_c15_s83.pt


## Evaluation

In [12]:
num_eval_eps = 500
nbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=nbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(nbc_returns)

Starting episode 1/500...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/500...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/500...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/500...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/500...


...

  Episode 497 ended at step 2000 (terminated: False, truncated: True).
Starting episode 498/500...


  Episode 498 ended at step 2000 (terminated: False, truncated: True).
Starting episode 499/500...


  Episode 499 ended at step 1009 (terminated: True, truncated: False).
Starting episode 500/500...


  Episode 500 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


925782

In [13]:
nbc_episode_rewards = defaultdict(float)
for rec in nbc_returns:
    ep = rec['episode']
    nbc_episode_rewards[ep] += float(rec['reward'])

nbc_rewards = [nbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(nbc_rewards) / num_eval_eps

-612.7004909643905

In [14]:
mean_reward = np.mean(nbc_rewards)
std_reward = np.std(nbc_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

E[Y]          = -612.7005
Std[Y]        = 202.6802
E[Y] ± Std[Y] = -612.7005 ± 202.6802


In [15]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in nbc_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

Success rate   = 19.40% (97/500 episodes)
Std error      = 1.77%


In [16]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")

Successful episode lengths (n=97):
  Mean   = 1234.87
  Std    = 395.67
  Median = 1214
  Min    = 408
  Max    = 1985
  25th%  = 941
  75th%  = 1549
